In [0]:
from datetime import datetime
import uuid

run_id = "uc16_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
task_name = "build_appointments_clean"

spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'STARTED',
  current_timestamp(),
  NULL,
  NULL,
  NULL
)
""")

In [0]:
%sql
CREATE OR REPLACE TABLE poc_health_no_show.bronze.appointments_raw
COMMENT 'Raw appointment data landed from Kaggle. Source fields retained for lineage and auditability.'
AS
SELECT
  CAST(PatientId AS STRING) AS patient_id,
  CAST(AppointmentID AS STRING) AS appointment_id,
  CAST(Gender AS STRING) AS gender,
  CAST(ScheduledDay AS STRING) AS scheduled_day,
  CAST(AppointmentDay AS STRING) AS appointment_day,
  CAST(Age AS STRING) AS age,
  CAST(Neighbourhood AS STRING) AS neighbourhood,
  CAST(Scholarship AS STRING) AS scholarship,
  CAST(Hipertension AS STRING) AS hipertension,
  CAST(Diabetes AS STRING) AS diabetes,
  CAST(Alcoholism AS STRING) AS alcoholism,
  CAST(Handcap AS STRING) AS handcap,
  CAST(`SMS_received` AS STRING) AS sms_received,
  CAST(`No-show` AS STRING) AS no_show,
  current_timestamp() AS _ingested_at,
  'appointments_raw.csv' AS _source_file,
  'kaggle_medical_appointment_no_shows' AS _source_system
FROM read_files(
  '/Volumes/poc_health_no_show/bronze/landing/appointments_raw.csv',
  format => 'csv',
  header => true,
  mode => 'FAILFAST'
);

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/poc_health_no_show/bronze/landing/appointments_raw.csv',
  format => 'csv',
  header => true,
  mode => 'FAILFAST'
)
LIMIT 5;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM poc_health_no_show.bronze.appointments_raw;

SELECT *
FROM poc_health_no_show.bronze.appointments_raw
LIMIT 20;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM poc_health_no_show.bronze.appointments_raw;

In [0]:
spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'SUCCESS',
  current_timestamp(),
  current_timestamp(),
  NULL,
  NULL
)
""")